In [7]:
import pandas as pd
import numpy as np
import csv
import pickle
import glob
import os
import ast
import matplotlib.pyplot as plt
from tqdm import tqdm
from print_color import print

### Models and Embeddings

In [8]:
def get_models(kg: str, emb_name: str, strategy: str, model: str, valid_rels: list):
    # Get models
    models = {}
    model_path = f"dumps_models/{kg}/{emb_name}/{strategy}/err_beta_1/{model}/*.pkl"

    for f in glob.glob(model_path):
        relation = f.split('/')[-1]
        start_pos = relation.find('_model_')
        relation = relation[start_pos + 7:]
        relation = relation[:-len('_fixed.pkl')] if relation.endswith('_fixed.pkl') else relation[:-len('.pkl')]
        if not relation in valid_rels: continue
        with open(f, "rb") as model:
            models[relation] = pickle.load(model)
    
    if not len(models): 
        raise Exception("No model is obtained! Check the file path")
    
    return models


In [ ]:
def get_input(kg: str, emb_name: str, strategy: str, needed_models: list, rels_pair: str, negative: bool):
    # Get samples & sample embeddings
    triples_dict = {}
    triples_cardinality = {}

    if not negative:
        dir_add = f"blind_test_occ/{kg}/*.csv"
    else:
        dir_add = f"negative_samples/{kg}/{strategy}/*.csv"
        # Get the number of blind sample triples for each relation
        blind_size = {}
        for f in glob.glob(f"blind_test_occ/{kg}/*.csv"):
            relation = f.split('/')[-1][:-4]
            if not relation in needed_models: continue
            blind_size[relation] = pd.read_csv(f).rename(columns={'subcject': 'subject'}).shape[0]

    for f in glob.glob(dir_add):
        relation = f.split('/')[-1][:-4]

        if not relation in needed_models: continue

        if not negative:
            triple_df = pd.read_csv(f).rename(columns={'subcject': 'subject'})
        else:
            # Get negative triples and sample them randomly with the corresponding blind sample size
            triple_df = pd.read_csv(f).rename(columns={'source': 'subject', 'target': 'object'})
            triple_df = triple_df.sample(n=min(blind_size[relation], len(triple_df)), random_state=5)

        if triple_df.shape[0] == 0:
            continue
        triples_dict[relation] = triple_df
        triples_cardinality[relation] = triples_dict[relation].shape[0]

    embedding = pd.read_csv(f"store_embeddings/{emb_name}/{kg}.csv")

    # Map triples to their embeddings
    mapped_emb = {}
    for rel in tqdm(triples_dict, desc="Mapping triples and create inputs"):
        mapped_emb[rel] = []
        for row in triples_dict[rel].iterrows():
            # Map URIs to embeddings
            uri = ''
            if emb_name == 'transe': uri = 'name'
            else: uri = 'node_id'

            # Find embedding vector from embedding list
            subject_emb = embedding[embedding[uri] == row[1]['subject']].squeeze().to_list()[1]
            object_emb = embedding[embedding[uri] == row[1]['object']].squeeze().to_list()[1]
            # Convert str to List
            subject_emb = ast.literal_eval(subject_emb)
            object_emb = ast.literal_eval(object_emb)
            # Compute model input
            mul_emb = np.array(subject_emb) * np.array(object_emb)
            mapped_emb[rel].append(mul_emb)

    if not negative:
        save_dir = f"formula_plausibility_files/{kg}/{kg}_blind_{rels_pair}.pkl"
    else:
        save_dir = f"formula_plausibility_files/{kg}/{kg}_{strategy}_{rels_pair}.pkl"
    print(save_dir)
    os.makedirs(os.path.dirname(save_dir), exist_ok=True)
    with open(save_dir, "wb") as f:
        pickle.dump(mapped_emb, f)
    
    return mapped_emb


### Helper Functions

In [10]:
# Helper Functions
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def s_p(rels_all, rel_org, results):
    """
        Calculates the relation of size of a target fact to sum of other facts
    """
    pred_size = int(results.loc[results['relation'] == rel_org, 'edges'].iloc[0])
    total_size = 0
    for rel in rels_all:
        total_size += int(results.loc[results['relation'] == rel, 'edges'].iloc[0]) 
    
    return pred_size / total_size

def ba_p(rels_all, rel_org, results):
    """
        Calculates the relation of Balanced Accuracy of a target model to sum of other models
    """
    pred_ba = float(results.loc[results['relation'] == rel_org, 'balanced_accuracy'].values[0][:5]) 
    total_ba = 0
    for rel in rels_all:
        total_ba += float(results.loc[results['relation'] == rel, 'balanced_accuracy'].values[0][:5])
    
    return pred_ba / total_ba

def w_p(s_p, ba_p):
    beta = 0.5
    return (1 + beta**2) * (s_p * ba_p) / ((beta**2 * s_p) + ba_p)

def delta_w(score_main, score_alt, w_main, w_alt):
    score_alt_w = {}
    for rel in list(score_alt.keys()):
        score_alt_w[rel] = score_alt[rel] * w_alt[rel]

    all_alt_scores = np.stack(list(score_alt_w.values()))
    d_w = (score_main * w_main) - np.max(all_alt_scores, axis=0) 
    return d_w  # d_w = [dw1, dw2, ...]


### Pbase

In [11]:
def p_base(model, samples):
    # probabilities for positive class
    pos_index = list(model.classes_).index(1)
    pos_probs = model.predict_proba(samples)[:, pos_index]
    return pos_probs    # pos_probs = [p1, p2, p3, ...] 

In [12]:
def p_base_dist(rel_main, score_main, score_alt):
    # All relations
    rels_all = []
    rels_all.append(rel_main)
    rels_all.extend(list(score_alt.keys()))
    # All scores
    score_alt[rel_main] = score_main

    p_score = {}
    for rel in rels_all:
        p_score[rel] = score_alt[rel]
    return p_score

### Pmax

In [13]:
# Plausibility Max Formula

def p_max(rel_main, rels_alt, score_main, score_alt, results, lmbda):

    rels_all = []
    rels_all.append(rel_main)
    rels_all.extend(rels_alt)
    
    # Main relation size and balanced accuracy
    s_p_main = s_p(rels_all, rel_main, results)
    ba_p_main = ba_p(rels_all, rel_main, results)
    w_main = w_p(s_p_main, ba_p_main)
    if not len(rels_alt):
        return sigmoid((lmbda * (score_main * w_main)))


    # Alternative relations size and balanced accuracy
    s_p_alt, ba_p_alt, w_alt = {}, {}, {}
    for rel_a in rels_alt:
        s_p_alt[rel_a] = s_p(rels_all, rel_a, results)
        ba_p_alt[rel_a] = ba_p(rels_all, rel_a, results)
        w_alt[rel_a] = w_p(s_p_alt[rel_a], ba_p_alt[rel_a])
    
    d_w = delta_w(score_main, score_alt, w_main, w_alt)
    p_score = sigmoid(lmbda * d_w)

    return p_score  # p_score = {alt_rel1: [p_scores1], alt_rel2: [p_scores2], ...}

In [14]:
def p_max_dist(rel_main, rels_alt, score_main, score_alt, results, lmbda):
    rels_all = []
    rels_all.append(rel_main)
    rels_all.extend(rels_alt)
    
    score_all = score_alt.copy() 
    score_all[rel_main] = score_main

    ps_scores = {}
    for temp_main in rels_all:
        rels_alt_temp = rels_all.copy()
        rels_alt_temp.remove(temp_main)
        score_alt_temp = {k:score_all[k] for k in rels_alt_temp}
        ps_scores[temp_main] = p_max(temp_main, rels_alt_temp, score_all[temp_main], score_alt_temp, results, lmbda)
    
    return ps_scores

### Psoftmax

In [15]:
def p_adj_softmax_2(rel_main, rels_alt, score_main, score_alt, results):
    lmbda = 10
    rels_all = []
    rels_all.append(rel_main)
    rels_all.extend(rels_alt)
    
    # Main relation size and balanced accuracy
    s_p_main = s_p(rels_all, rel_main, results)
    ba_p_main = ba_p(rels_all, rel_main, results)
    w_main = w_p(s_p_main, ba_p_main)

    numerator = np.exp(lmbda * score_main * w_main)
    # No alternative relation case
    if len(rels_alt) == 0:
        return score_main * w_main

    # Alternative relations size and balanced accuracy
    s_p_alt, ba_p_alt, w_alt = {}, {}, {}
    for rel_a in rels_alt:
        s_p_alt[rel_a] = s_p(rels_all, rel_a, results)
        ba_p_alt[rel_a] = ba_p(rels_all, rel_a, results)
        w_alt[rel_a] = w_p(s_p_alt[rel_a], ba_p_alt[rel_a])
    # 1 alternative relation case
    if len(rels_alt) == 1:
        alt_score = list(score_alt.values())[0]
        alt_score_w = alt_score * w_alt[list(score_alt.keys())[0]]
        denominator = numerator + np.exp(lmbda * alt_score_w) 
        return numerator / denominator

    # Multiple alternative relations case
    alt_score_w = {}
    for rel_a in rels_alt:
        alt_score_w[rel_a] = score_alt[rel_a] * w_alt[rel_a]

    stacked_alt_score_w = np.stack(list(alt_score_w.values()))
    score_best_alt_w = np.max(stacked_alt_score_w, axis=0)
    score_2nd_best_alt_w = np.partition(stacked_alt_score_w, -2, axis=0)[-2]
    denominator = numerator + np.exp(lmbda * score_best_alt_w) + np.exp(lmbda * score_2nd_best_alt_w)
     
    return numerator / denominator

In [16]:
def p_adj_softmax_1(rel_main, rels_alt, score_main, score_alt, results):
    lmbda = 10
    rels_all = []
    rels_all.append(rel_main)
    rels_all.extend(rels_alt)

    # Main relation size and balanced accuracy
    s_p_main = s_p(rels_all, rel_main, results)
    ba_p_main = ba_p(rels_all, rel_main, results)
    w_main = w_p(s_p_main, ba_p_main)

    numerator = np.exp(lmbda * score_main * w_main)
    # No alternative relation case
    if len(rels_alt) == 0:
        return score_main * w_main

    # Alternative relations size and balanced accuracy
    s_p_alt, ba_p_alt, w_alt = {}, {}, {}
    for rel_a in rels_alt:
        s_p_alt[rel_a] = s_p(rels_all, rel_a, results)
        ba_p_alt[rel_a] = ba_p(rels_all, rel_a, results)
        w_alt[rel_a] = w_p(s_p_alt[rel_a], ba_p_alt[rel_a])

    alt_score_w = {}
    for rel_a in rels_alt:
        alt_score_w[rel_a] = score_alt[rel_a] * w_alt[rel_a]

    stacked_alt_score_w = np.stack(list(alt_score_w.values()))
    score_best_alt_w = np.max(stacked_alt_score_w, axis=0)
    denominator = numerator + np.exp(lmbda * score_best_alt_w)

    return numerator / denominator

In [17]:
def p_adj_softmax_3(rel_main, rels_alt, score_main, score_alt, results):
    lmbda = 10
    rels_all = []
    rels_all.append(rel_main)
    rels_all.extend(rels_alt)

    s_p_main = s_p(rels_all, rel_main, results)
    ba_p_main = ba_p(rels_all, rel_main, results)
    w_main = w_p(s_p_main, ba_p_main)

    numerator = np.exp(lmbda * score_main * w_main)
    # No alternative relation case
    if len(rels_alt) == 0:
        return score_main * w_main

    # Alternative relations size and balanced accuracy
    s_p_alt, ba_p_alt, w_alt = {}, {}, {}
    for rel_a in rels_alt:
        s_p_alt[rel_a] = s_p(rels_all, rel_a, results)
        ba_p_alt[rel_a] = ba_p(rels_all, rel_a, results)
        w_alt[rel_a] = w_p(s_p_alt[rel_a], ba_p_alt[rel_a])

    alt_score_w = {}
    for rel_a in rels_alt:
        alt_score_w[rel_a] = score_alt[rel_a] * w_alt[rel_a]

    stacked_alt_score_w = np.stack(list(alt_score_w.values()))
    score_best_alt_w = np.max(stacked_alt_score_w, axis=0)

    # 1 alternative relation case
    if len(rels_alt) == 1:
        denominator = numerator + np.exp(lmbda * score_best_alt_w)
        return numerator / denominator

    score_2nd_best_alt_w = np.partition(stacked_alt_score_w, -2, axis=0)[-2]

    # 2 alternative relations case
    if len(rels_alt) == 2:
        denominator = numerator + np.exp(lmbda * score_best_alt_w) + np.exp(lmbda * score_2nd_best_alt_w)
        return numerator / denominator

    # More than 2 alternative relations case
    score_3rd_best_alt_w = np.partition(stacked_alt_score_w, -3, axis=0)[-3]
    denominator = numerator + np.exp(lmbda * score_best_alt_w) + np.exp(lmbda * score_2nd_best_alt_w) + np.exp(lmbda * score_3rd_best_alt_w)

    return numerator / denominator

In [18]:
def p_adj_softmax_1_dist(rel_main, rels_alt, score_main, score_alt, results):
    rels_all = []
    rels_all.append(rel_main)
    rels_all.extend(rels_alt)

    score_all = score_alt.copy()
    score_all[rel_main] = score_main

    ps_scores = {}
    for temp_main in rels_all:
        rels_alt_temp = rels_all.copy()
        rels_alt_temp.remove(temp_main)
        score_alt_temp = {k: score_all[k] for k in rels_alt_temp}
        ps_scores[temp_main] = p_adj_softmax_1(temp_main, rels_alt_temp, score_all[temp_main], score_alt_temp, results)

    return ps_scores

In [19]:
def p_adj_softmax_3_dist(rel_main, rels_alt, score_main, score_alt, results):
    rels_all = []
    rels_all.append(rel_main)
    rels_all.extend(rels_alt)

    score_all = score_alt.copy()
    score_all[rel_main] = score_main

    ps_scores = {}
    for temp_main in rels_all:
        rels_alt_temp = rels_all.copy()
        rels_alt_temp.remove(temp_main)
        score_alt_temp = {k: score_all[k] for k in rels_alt_temp}
        ps_scores[temp_main] = p_adj_softmax_3(temp_main, rels_alt_temp, score_all[temp_main], score_alt_temp, results)

    return ps_scores

In [20]:
def p_adj_softmax_2_dist(rel_main, rels_alt, score_main, score_alt, results):
    rels_all = []
    rels_all.append(rel_main)
    rels_all.extend(rels_alt)
    
    score_all = score_alt.copy() 
    score_all[rel_main] = score_main

    ps_scores = {}
    for temp_main in rels_all:
        rels_alt_temp = rels_all.copy()
        rels_alt_temp.remove(temp_main)
        score_alt_temp = {k:score_all[k] for k in rels_alt_temp}
        ps_scores[temp_main] = p_adj_softmax_2(temp_main, rels_alt_temp, score_all[temp_main], score_alt_temp, results)
    
    return ps_scores

### Comb

In [21]:
def p_comb(rel_main, rels_alt, score_main, score_alt, results, max_lmbda=10.0):
    p_max_scores = p_max(rel_main, rels_alt, score_main, score_alt, results, lmbda=max_lmbda)
    return (0.5 * score_main) + (0.5 * p_max_scores)

def p_comb_2(rel_main, rels_alt, score_main, score_alt, results, max_lmbda=10.0):
    p_max_scores = p_max(rel_main, rels_alt, score_main, score_alt, results, lmbda=max_lmbda)
    p_softmax_scores = p_adj_softmax_2(rel_main, rels_alt, score_main, score_alt, results)
    return (0.5 * score_main) + (0.25 * p_max_scores) + (0.25 * p_softmax_scores)

def p_comb_3(rel_main, rels_alt, score_main, score_alt, results, max_lmbda=10.0):
    p_max_scores = p_max(rel_main, rels_alt, score_main, score_alt, results, lmbda=max_lmbda)
    p_softmax_scores = p_adj_softmax_2(rel_main, rels_alt, score_main, score_alt, results)
    return (0.333 * score_main) + (0.333 * p_max_scores) + (0.333 * p_softmax_scores)

def p_comb_4(rel_main, rels_alt, score_main, score_alt, results, max_lmbda=10.0):
    p_max_scores = p_max(rel_main, rels_alt, score_main, score_alt, results, lmbda=max_lmbda)
    p_softmax_scores = p_adj_softmax_2(rel_main, rels_alt, score_main, score_alt, results)
    return (0.25 * score_main) + (0.375 * p_max_scores) + (0.375 * p_softmax_scores)

def p_comb_5(rel_main, rels_alt, score_main, score_alt, results, max_lmbda=10.0):
    p_max_scores = p_max(rel_main, rels_alt, score_main, score_alt, results, lmbda=max_lmbda)
    p_softmax_scores = p_adj_softmax_2(rel_main, rels_alt, score_main, score_alt, results)
    return (0.75 * score_main) + (0.125 * p_max_scores) + (0.125 * p_softmax_scores)

def p_comb_6(rel_main, rels_alt, score_main, score_alt, results):
    p_soft_scores = p_adj_softmax_2(rel_main, rels_alt, score_main, score_alt, results)
    return (0.5 * score_main) + (0.5 * p_soft_scores)

def p_comb_7(rel_main, rels_alt, score_main, score_alt, results, max_lmbda=10.0):
    p_max_scores = p_max(rel_main, rels_alt, score_main, score_alt, results, lmbda=max_lmbda)
    return (0.25 * score_main) + (0.75 * p_max_scores)

def p_comb_8(rel_main, rels_alt, score_main, score_alt, results, max_lmbda=10.0):
    p_max_scores = p_max(rel_main, rels_alt, score_main, score_alt, results, lmbda=max_lmbda)
    return (0.75 * score_main) + (0.25 * p_max_scores)

In [22]:
def p_comb_dist(rel_main, rels_alt, score_main, score_alt, results, max_lmbda=10.0):
    rels_all = []
    rels_all.append(rel_main)
    rels_all.extend(rels_alt)
    
    score_all = score_alt.copy() 
    score_all[rel_main] = score_main

    ps_scores = {}
    for temp_main in rels_all:
        rels_alt_temp = rels_all.copy()
        rels_alt_temp.remove(temp_main)
        score_alt_temp = {k:score_all[k] for k in rels_alt_temp}
        ps_scores[temp_main] = p_comb(temp_main, rels_alt_temp, score_all[temp_main], score_alt_temp, results, max_lmbda)
    
    return ps_scores

def p_comb_2_dist(rel_main, rels_alt, score_main, score_alt, results, max_lmbda=10.0):
    rels_all = [rel_main] + rels_alt
    score_all = score_alt.copy()
    score_all[rel_main] = score_main
    ps_scores = {}
    for temp_main in rels_all:
        rels_alt_temp = [r for r in rels_all if r != temp_main]
        score_alt_temp = {k: score_all[k] for k in rels_alt_temp}
        ps_scores[temp_main] = p_comb_2(temp_main, rels_alt_temp, score_all[temp_main], score_alt_temp, results, max_lmbda)
    return ps_scores

def p_comb_3_dist(rel_main, rels_alt, score_main, score_alt, results, max_lmbda=10.0):
    rels_all = [rel_main] + rels_alt
    score_all = score_alt.copy()
    score_all[rel_main] = score_main
    ps_scores = {}
    for temp_main in rels_all:
        rels_alt_temp = [r for r in rels_all if r != temp_main]
        score_alt_temp = {k: score_all[k] for k in rels_alt_temp}
        ps_scores[temp_main] = p_comb_3(temp_main, rels_alt_temp, score_all[temp_main], score_alt_temp, results, max_lmbda)
    return ps_scores

def p_comb_4_dist(rel_main, rels_alt, score_main, score_alt, results, max_lmbda=10.0):
    rels_all = [rel_main] + rels_alt
    score_all = score_alt.copy()
    score_all[rel_main] = score_main
    ps_scores = {}
    for temp_main in rels_all:
        rels_alt_temp = [r for r in rels_all if r != temp_main]
        score_alt_temp = {k: score_all[k] for k in rels_alt_temp}
        ps_scores[temp_main] = p_comb_4(temp_main, rels_alt_temp, score_all[temp_main], score_alt_temp, results, max_lmbda)
    return ps_scores

def p_comb_5_dist(rel_main, rels_alt, score_main, score_alt, results, max_lmbda=10.0):
    rels_all = [rel_main] + rels_alt
    score_all = score_alt.copy()
    score_all[rel_main] = score_main
    ps_scores = {}
    for temp_main in rels_all:
        rels_alt_temp = [r for r in rels_all if r != temp_main]
        score_alt_temp = {k: score_all[k] for k in rels_alt_temp}
        ps_scores[temp_main] = p_comb_5(temp_main, rels_alt_temp, score_all[temp_main], score_alt_temp, results, max_lmbda)
    return ps_scores

def p_comb_6_dist(rel_main, rels_alt, score_main, score_alt, results):
    rels_all = [rel_main] + rels_alt
    score_all = score_alt.copy()
    score_all[rel_main] = score_main
    ps_scores = {}
    for temp_main in rels_all:
        rels_alt_temp = [r for r in rels_all if r != temp_main]
        score_alt_temp = {k: score_all[k] for k in rels_alt_temp}
        ps_scores[temp_main] = p_comb_6(temp_main, rels_alt_temp, score_all[temp_main], score_alt_temp, results)
    return ps_scores

def p_comb_7_dist(rel_main, rels_alt, score_main, score_alt, results, max_lmbda=10.0):
    rels_all = [rel_main] + rels_alt
    score_all = score_alt.copy()
    score_all[rel_main] = score_main
    ps_scores = {}
    for temp_main in rels_all:
        rels_alt_temp = [r for r in rels_all if r != temp_main]
        score_alt_temp = {k: score_all[k] for k in rels_alt_temp}
        ps_scores[temp_main] = p_comb_7(temp_main, rels_alt_temp, score_all[temp_main], score_alt_temp, results, max_lmbda=max_lmbda)
    return ps_scores

def p_comb_8_dist(rel_main, rels_alt, score_main, score_alt, results, max_lmbda=10.0):
    rels_all = [rel_main] + rels_alt
    score_all = score_alt.copy()
    score_all[rel_main] = score_main
    ps_scores = {}
    for temp_main in rels_all:
        rels_alt_temp = [r for r in rels_all if r != temp_main]
        score_alt_temp = {k: score_all[k] for k in rels_alt_temp}
        ps_scores[temp_main] = p_comb_8(temp_main, rels_alt_temp, score_all[temp_main], score_alt_temp, results, max_lmbda=max_lmbda)
    return ps_scores   

### Get Score

In [23]:
def get_score(kg: str, embeddings, models, formula, results, max_lmbda):
    relations = list(embeddings.keys())
    rel_parts = {}
    for rel in relations:
        sub, pred, obj = rel.split(' - ')
        rel_parts[rel] = {'subject': sub, 'object': obj, 'predicate': pred}

    plausibility = {}
    for rel in relations:
        plausibility[rel] = {}

        # Compute alternative relations
        sub, obj, pred = rel_parts[rel]['subject'],rel_parts[rel]['object'],rel_parts[rel]['predicate']
        preds_alt = [rel_parts[k]['predicate'] for k in rel_parts if rel_parts[k]['subject'] == sub and rel_parts[k]['object'] == obj]
        rels_alt = [f'{sub} - {p} - {obj}' for p in preds_alt] # includes main relation
        rels_alt.remove(rel)

        model_main = models[rel]
        embeddings_main = embeddings[rel]
        # get model's score
        score_main = p_base(model_main, embeddings_main) 
        
        score_alt = {}
        for rel_a in rels_alt:
            model_alt = models[rel_a]
            # get model's score
            score_alt[rel_a] = p_base(model_alt, embeddings_main)
        if formula == 'max':
            plausibility[rel] = p_max(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'max_dist':
            plausibility[rel] = p_max_dist(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'adj_softmax_2':
            plausibility[rel] = p_adj_softmax_2(rel, rels_alt, score_main, score_alt, results)
        elif formula == 'adj_softmax_2_dist':
            plausibility[rel] = p_adj_softmax_2_dist(rel, rels_alt, score_main, score_alt, results)
        elif formula == 'adj_softmax_1':
            plausibility[rel] = p_adj_softmax_1(rel, rels_alt, score_main, score_alt, results)
        elif formula == 'adj_softmax_1_dist':
            plausibility[rel] = p_adj_softmax_1_dist(rel, rels_alt, score_main, score_alt, results)
        elif formula == 'adj_softmax_3':
            plausibility[rel] = p_adj_softmax_3(rel, rels_alt, score_main, score_alt, results)
        elif formula == 'adj_softmax_3_dist':
            plausibility[rel] = p_adj_softmax_3_dist(rel, rels_alt, score_main, score_alt, results)
        elif formula == 'comb':
            plausibility[rel] = p_comb(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'comb_2':
            plausibility[rel] = p_comb_2(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'comb_3':
            plausibility[rel] = p_comb_3(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'comb_4':
            plausibility[rel] = p_comb_4(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'comb_5':
            plausibility[rel] = p_comb_5(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'comb_6':
            plausibility[rel] = p_comb_6(rel, rels_alt, score_main, score_alt, results)
        elif formula == 'comb_7':
            plausibility[rel] = p_comb_7(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'comb_8':
            plausibility[rel] = p_comb_8(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'comb_dist':
            plausibility[rel] = p_comb_dist(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'comb_2_dist':
            plausibility[rel] = p_comb_2_dist(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'comb_3_dist':
            plausibility[rel] = p_comb_3_dist(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'comb_4_dist':
            plausibility[rel] = p_comb_4_dist(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'comb_5_dist':
            plausibility[rel] = p_comb_5_dist(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'comb_6_dist':
            plausibility[rel] = p_comb_6_dist(rel, rels_alt, score_main, score_alt, results)
        elif formula == 'comb_7_dist':
            plausibility[rel] = p_comb_7_dist(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'comb_8_dist':
            plausibility[rel] = p_comb_8_dist(rel, rels_alt, score_main, score_alt, results, max_lmbda)
        elif formula == 'base':
            plausibility[rel] = score_main
        elif formula == 'base_dist':
            plausibility[rel] = p_base_dist(rel, score_main, score_alt)
        else:
            raise Exception('Formula name does not exist. ("max", "soft", "comb", "max_dist", "comb_dist", "base", "base_dist")')

    return plausibility


In [24]:
def run(kg: str, 
        emb_name: str, 
        strategy: str, 
        model: str, 
        formula: str, 
        results: pd.DataFrame,
        rels_pair: str, 
        valid_rels: list,
        negative: bool,
        max_lmbda: float):

    models = get_models(kg=kg, emb_name=emb_name, strategy=strategy, model=model, valid_rels=valid_rels)
    models_name = list(models.keys())
    
    # blind_embs = None
    if not negative:
        blind_path = f"formula_plausibility_files/{kg}/{kg}_blind_{rels_pair}.pkl"
        if not os.path.exists(blind_path):
            triple_embs = get_input(kg=kg, 
                                    emb_name=emb_name, 
                                    strategy=strategy, 
                                    needed_models=models_name, 
                                    rels_pair=rels_pair,
                                    negative=negative)
        else:
            with open(blind_path, 'rb') as f:
                triple_embs = pickle.load(f)
    else:
        neg_path = f"formula_plausibility_files/{kg}/{kg}_{strategy}_{rels_pair}.pkl"
        if not os.path.exists(neg_path):
            triple_embs = get_input(kg=kg, 
                                    emb_name=emb_name, 
                                    strategy=strategy, 
                                    needed_models=models_name, 
                                    rels_pair=rels_pair,
                                    negative=negative)
        else:
            with open(neg_path, 'rb') as f:
                triple_embs = pickle.load(f)

    relations = list(triple_embs.keys())
    # subset of models that their input exist
    models_subset = {k: models[k] for k in list(models.keys()) if k in relations}
    # subset of triple_embs that their model exist
    triple_embs = {k: triple_embs[k] for k in relations if k in models_subset}
    
    return get_score(kg=kg, embeddings=triple_embs, models=models_subset, formula=formula, results=results, max_lmbda=max_lmbda)


### KG Relations

#### miRNAdiseaseGO

In [ ]:
miRNA_KG_relations = {
    'Gene - Disease': [
        'Gene - causes or contributes to condition - Disease',
        ],
    'miRNA - miRNA': [
        'miRNA - in similarity relationship with - miRNA',
        ],
    'miRNA - Disease': [
        'miRNA - causes or contributes to condition - Disease',
        'miRNA - under expressed in - Disease',
        'miRNA - over expressed in - Disease',
        'miRNA - is causal somatic mutation in - Disease',
        ],
    'Gene - Gene': [
        'Gene - genetically interacts with - Gene',
        ],
    'miRNA - Gene': [
        'miRNA - regulates activity of - Gene',
        ],
    'miRNA - GO': [
        'miRNA - participates in - GO',
        'miRNA - has function - GO',
        'miRNA - located in - GO',
        'miRNA - part of - GO',
        ]
    }


#### diseasemech

In [ ]:
PKT_KG_relations = {
    'Protein - Anatomy': [
        'Protein - located in - Anatomy'
        ],
    'Protein - Cell': [
        'Protein - located in - Cell'
        ],
    'Protein - Protein': [    
        'Protein - molecularly interacts with - Protein'
        ],
    'GO - GO': [
        'GO - negatively regulates - GO', 
        'GO - positively regulates - GO', 
        'GO - regulates - GO',
        ],
    'Chemical - GO': [
        'Chemical - participates in - GO',
        'Chemical - molecularly interacts with - GO',
        ],
    'Protein - GO': [
        'Protein - enables - GO',
        'Protein - located in - GO',
        ],
    'Chemical - Gene': [
        'Chemical - interacts with - Gene',
        ],
    'Chemical - Protein': [
        'Chemical - interacts with - Protein',
        'Chemical - molecularly interacts with - Protein'
        ],
    'Chemical - Disease': [
        'Chemical - is substance that treats - Disease'
        ],
    'Chemical - Pathway': [
        'Chemical - participates in - Pathway'
        ],
    'Protein - Pathway': [
        'Protein - participates in - Pathway'
        ],
    'Gene - Disease': [
        'Gene - causes or contributes to condition - Disease',
        ],
    'Gene - Gene': [
        'Gene - genetically interacts with - Gene'
        ],
    'Gene - Protein': [
        'Gene - interacts with - Protein'
        ],
    'Gene - Pathway': [
        'Gene - participates in - Pathway'
        ]
}


#### hetionet

In [ ]:
Hetionet_relations = {
    'Anatomy - Gene': [
        'Anatomy - downregulates - Gene',
        'Anatomy - expresses - Gene', 
        'Anatomy - upregulates - Gene',
        ],
    'Compund - Gene': [
        'Compound - binds - Gene',
        'Compound - downregulates - Gene',
        'Compound - upregulates - Gene',
        ],
    'Compound - Side_effect': [
        'Compound - causes - Side_effect', 
        ],
    'Compound - Compound': [
        'Compound - resembles - Compound',
        ],
    'Disease - Gene': [
        'Disease - associates - Gene',
        'Disease - downregulates - Gene',
        'Disease - upregulates - Gene',
        ],
    'Disease - Anatomy': [
        'Disease - localizes - Anatomy',
        ],
    'Disease - Symptom': [
        'Disease - presents - Symptom', 
        ],
    'Gene - Gene': [
        'Gene - covaries - Gene',
        'Gene - interacts - Gene', 
        'Gene - regulates - Gene', 
        ],
    'Gene - Biological_process': [
        'Gene - participates - Biological_process',
        ],
    'Gene - Cellular_component': [
        'Gene - participates - Cellular_component',
        ],
    'Gene - Molecular_function': [
        'Gene - participates - Molecular_function',
        ],
    'Gene - Pathway': [
        'Gene - participates - Pathway', 
        ],
    'Pharmacologic_class  - Compound': [
        'Pharmacologic_class - includes - Compound'
        ]
}


#### primeKG

In [ ]:
PrimeKG_relations = {
    'Anatomy - Gene_and_or_protein': [
        'Anatomy - expression absent - Gene_and_or_protein',
        'Anatomy - expression present - Gene_and_or_protein', 
        ],
    'Biological_process - Exposure': [
        'Biological_process - interacts with - Exposure', 
        ],
    'Biological_process - Gene_and_or_protein': [
        'Biological_process - interacts with - Gene_and_or_protein',
        ],
    'Cellular_component - Gene_and_or_protein' : [
        'Cellular_component - interacts with - Gene_and_or_protein',
        ],
    'Disease - Gene_and_or_protein' : [
        'Disease - associated with - Gene_and_or_protein',
        ],
    'Disease - Drug': [
        'Disease - contraindication - Drug',
        'Disease - indication - Drug',
        'Disease - off label use - Drug',
        ],
    'Disease - Exposure': [
        'Disease - linked to - Exposure',
        ],
    'Disease - Effect_and_or_phenotype': [
        'Disease - phenotype absent - Effect_and_or_phenotype',
        'Disease - phenotype present - Effect_and_or_phenotype', 
        ],
    'Drug - Gene_and_or_protein': [
        'Drug - enzyme - Gene_and_or_protein',
        'Drug - target - Gene_and_or_protein',
        'Drug - transporter - Gene_and_or_protein',
        ],
    'Drug - Effect_and_or_phenotype': [
        'Drug - side effect - Effect_and_or_phenotype',
        ],
    'Drug - Drug': [
        'Drug - synergistic interaction - Drug', 
        ],
    'Effect_and_or_phenotype - Gene_and_or_protein': [
        'Effect_and_or_phenotype - associated with - Gene_and_or_protein',
        ],
    'Exposure - Gene_and_or_protein': [
        'Exposure - interacts with - Gene_and_or_protein', 
        ],
    'Gene_and_or_protein - Molecular_function': [
        'Gene_and_or_protein - interacts with - Molecular_function',
        ],
    'Gene_and_or_protein - Pathway': [
        'Gene_and_or_protein - interacts with - Pathway',
        ],
    'Gene_and_or_protein - Gene_and_or_protein': [
        'Gene_and_or_protein - ppi - Gene_and_or_protein' 
        ],
}


#### OptimusKG

In [29]:
OptimusKG_relations = {
    'Anatomy - Gene': [
        'Anatomy - EXPRESSION_ABSENT - Gene',               
        'Anatomy - EXPRESSION_PRESENT - Gene',              
        ],
    'Drug - Phenotype': [
        'Drug - ASSOCIATED_WITH - Phenotype',
        'Drug - CONTRAINDICATION - Phenotype',          
        'Drug - INDICATION - Phenotype',                
        ],         
    'Drug - Disease': [
        'Drug - OFF_LABEL_USE - Disease',               
        'Drug - CONTRAINDICATION - Disease',            
        'Drug - INDICATION - Disease',                  
        ],
    'Drug - Gene': [
        'Drug - TARGET - Gene',                         
        'Drug - TRANSPORTER - Gene',                    
        'Drug - ENZYME - Gene',                         
        ],
    'Biological_process - Gene': [
        'Biological_process - INTERACTS_WITH - Gene',
        ],
    'Cellular_component - Gene': [
        'Cellular_component - INTERACTS_WITH - Gene',   
        ],
    'Disease - Gene': [
        'Disease - ASSOCIATED_WITH - Gene',             
        ],
    'Pathway - Gene': [
        'Pathway - INTERACTS_WITH - Gene',              
        ],
    'Disease - Phenotype': [
        'Disease - PHENOTYPE_PRESENT - Phenotype',      
        ],
    'Drug - Drug': [
        'Drug - SYNERGISTIC_INTERACTION - Drug',            
        ],
    'Exposure - Biological_process': [
        'Exposure - INTERACTS_WITH - Biological_process'
        ],
    'Exposure - Gene': [
        'Exposure - INTERACTS_WITH - Gene',             
        ],
    'Exposure - Disease': [
        'Exposure - LINKED_TO - Disease',               
        ],
    'Phenotype - Gene': [   
        'Phenotype - ASSOCIATED_WITH - Gene',
        ],
    'Gene - Gene': [
        'Gene - INTERACTS_WITH - Gene',                 
        ],
    'Molecular_function - Gene': [ 
        'Molecular_function - INTERACTS_WITH - Gene'    
        ]
}


In [ ]:
def create_plots(p_scores: dict, kg: str, negative: bool, strategy: str, rels_pair: str, model: str, formula: str):
    if negative:
        color = 'r'
        samples = 'Negative Samples'
    else:
        color = 'g'
        samples = 'Blind Test'
    f = plt.figure(figsize=(12, 12))
    plt.style.use('seaborn-v0_8')
    sp = 1

    if len(list(p_scores.keys())) > 1:
        for m_rel in p_scores.keys():
            alt_rels = p_scores[m_rel].keys()
            for a_rel in alt_rels:
                plt.subplot(len(p_scores.keys()), len(alt_rels), sp)
                # plt.subplot(len(alt_rels), len(p_scores.keys()), sp)
                plt.hist(p_scores[m_rel][a_rel], bins=40)
                plt.grid(True)
                plt.title(m_rel, c=color)
                plt.xlabel(a_rel)
                sp +=1
        plt.suptitle(f"{formula.upper()} Plausibility Formula on {samples} Community {model} ({rels_pair})", fontsize=20, fontweight='bold', y=0.92)
        image_ST = 'negative' if negative else 'blind'
        image_strat = 'comm' if strategy == 'c-b-n-s' else 'rndm'
        plt.savefig(f'plot_results/{model}/{kg}/{strategy}/{rels_pair}_{formula}_{image_ST}_sample_{image_strat}_{model}_fixed', dpi=100)
        plt.show()

    else:
        for rel in p_scores.keys():
            plt.subplot(len(p_scores.keys()), 1, sp)
            plt.hist(p_scores[rel], bins=40)
            plt.grid(True)
            plt.title(rel, c=color)
            sp += 1
        plt.suptitle(f"{formula.upper()} Plausibility Formula on {samples} Community {model} ({rels_pair})", fontsize=20, fontweight='bold', y=0.92)
        image_ST = 'negative' if negative else 'blind'
        image_strat = 'comm' if strategy == 'c-b-n-s' else 'rndm'
        # plt.savefig(f'plot_results/{model}/{kg}/{strategy}/{rels_pair}_{formula}_{image_ST}_sample_{image_strat}_{model}', dpi=100)
        plt.show()


In [ ]:
def _experiments_csv(kg, emb, strategy, model='RF'):
    for suffix in ['', '_fixed']:
        path = f'experiments/{kg}/{emb}/{model}_{strategy}{suffix}.csv'
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"No experiments csv found for {kg}/{emb}/{strategy}")

def exe_ps_p(kg: str, rels_pair: str, lmbda: float):
    # Compute Plausibility Scores and Plot Histograms
    if kg == 'miRNA-KG':
        valid_rels = miRNA_KG_relations[rels_pair]
    elif kg == 'PKT-KG':
        valid_rels = PKT_KG_relations[rels_pair]
    elif kg == 'Hetionet':
        valid_rels = Hetionet_relations[rels_pair]
    elif kg == 'PrimeKG':
        valid_rels = PrimeKG_relations[rels_pair]

    strategy = 'c-b-n-s'
    emb = 'transe'
    results = pd.read_csv(_experiments_csv(kg, emb, strategy))[['relation', 'edges', 'balanced_accuracy']]

    model = 'RF'

    negatives = [False, True]
    formulas = ['comb']
    for neg in negatives:
        for frml in formulas:

            p = run(kg=kg, 
                    emb_name=emb, 
                    strategy=strategy, 
                    model=model, 
                    formula=frml, 
                    results=results, 
                    rels_pair=rels_pair, 
                    valid_rels=valid_rels,
                    negative=neg,
                    lmbda=lmbda
                    )
            create_plots(p_scores=p,
                            kg=kg, 
                            negative=neg, 
                            strategy=strategy,
                            rels_pair=rels_pair,
                            model=model,
                            formula=frml)


In [ ]:
def exe_ps(kg: str, rels_pair: str, strategy: str, formula: str, negative: bool, max_lmbda: float):
    # Compute Plausibility Scores and Plot Histograms
    if kg == 'miRNA-KG':
        valid_rels = miRNA_KG_relations[rels_pair]
    elif kg == 'PKT-KG':
        valid_rels = PKT_KG_relations[rels_pair]
    elif kg == 'Hetionet':
        valid_rels = Hetionet_relations[rels_pair]
    elif kg == 'PrimeKG':
        valid_rels = PrimeKG_relations[rels_pair]
    elif kg == 'OptimusKG':
        valid_rels = OptimusKG_relations[rels_pair]

    emb = 'transe'
    results = pd.read_csv(_experiments_csv(kg, emb, strategy))[['relation', 'edges', 'balanced_accuracy']]

    model = 'RF'
    return run(
        kg=kg, 
        emb_name=emb, 
        strategy=strategy, 
        model=model, 
        formula=formula, 
        results=results, 
        rels_pair=rels_pair, 
        valid_rels=valid_rels,
        negative=negative,
        max_lmbda=max_lmbda,
        )


### Metrics

In [30]:
def global_mean_std(dict_list: list, negative: bool):
    all_values = []
    all_values_keys = {}
    for scores_dict in dict_list:
        for rel in scores_dict.keys():
                if negative:
                    all_values.append(np.abs(scores_dict[rel]))
                    all_values_keys[rel] = np.abs(scores_dict[rel])
                else:
                    all_values.append(np.abs(1 - scores_dict[rel]))
                    all_values_keys[rel] = np.abs(1 - scores_dict[rel])
    
    
    all_values = np.concatenate(all_values)
    return np.mean(all_values), np.std(all_values), all_values_keys

In [31]:
def global_mean_std_combined(pos_dict_list, neg_dict_list):
    all_values = []
    all_values_keys = {}
    for scores_dict in pos_dict_list:
        for rel in scores_dict.keys():
                all_values.append(np.abs(1 - scores_dict[rel]))
                all_values_keys[rel] = np.abs(1 - scores_dict[rel])

    for scores_dict in neg_dict_list:
        for rel in scores_dict.keys():
                all_values.append(np.abs(scores_dict[rel]))
                pos_err = all_values_keys[rel]
                neg_err = np.abs(scores_dict[rel])
                if len(pos_err) == len(neg_err):
                    all_values_keys[rel] = np.stack([pos_err, neg_err])
                else:
                    n = min(len(pos_err), len(neg_err))
                    all_values_keys[rel] = np.mean(np.stack([pos_err[:n], neg_err[:n]]), axis=0).mean()
    
    all_values_keys = {k: np.mean(v, axis=0) if np.ndim(v) > 0 else v for k, v in all_values_keys.items()}
    all_values = np.concatenate(all_values)
    return np.mean(all_values), np.std(all_values), all_values_keys


In [32]:
def global_mean_std_main_vs_alts(dict_list):
    all_values = []
    all_values_keys = {} 
    for scores_dict in dict_list:
        for main_key, inner_dict in scores_dict.items():
            all_values_keys[main_key] = {}
            main_arr = inner_dict[main_key]
            for alt_key, alt_arr in inner_dict.items():
                if alt_key == main_key:
                    continue
                all_values.append(np.abs(main_arr - alt_arr))
                all_values_keys[main_key][alt_key] = np.abs(main_arr - alt_arr)
        
        if not all_values_keys[main_key]:
            del all_values_keys[main_key]
    
    all_values = np.concatenate(all_values)
    return np.mean(all_values), np.std(all_values), all_values_keys

In [33]:
def avg_sign(dict_list: list, alt: str, threshold: dict = None):
    # save signs
    all_signes = []
    all_values_keys = {}
    # loop through each pair
    scores_used = {'P_correct': {}, 'P_best': {}, 'P_worst': {}, 'threshold_best': {}, 'threshold_worst': {}}
    scores_used_mean = {'P_correct': {}, 'P_best': {}, 'P_worst': {}, 'threshold_best': {}, 'threshold_worst': {}}
    for scores_dict in dict_list:
        # loop through each relation
        for key, inner_dict in scores_dict.items():
            main_key = key
            filtered_inner_dict = dict(filter(lambda i: i[0] != main_key, inner_dict.items()))

            if not filtered_inner_dict:
                continue

            # get correct, best and worst scores
            scores_stack = np.stack(list(filtered_inner_dict.values()))
            scores_used['P_correct'][main_key] = inner_dict[main_key]
            scores_used['P_best'][main_key] = np.max(scores_stack, axis=0)
            scores_used['P_worst'][main_key] = np.min(scores_stack, axis=0)

            tsh = 0.5   # default threshold
            if threshold != None:
                tsh = threshold[main_key]
            # assign signs 
            signed_main = [0 if s > tsh else 1 for s in scores_used['P_correct'][main_key]]
            if alt == 'best':
                signed_alt = [0 if s > tsh else 1 for s in scores_used['P_best'][main_key]]
            elif alt == 'worst':
                signed_alt = [0 if s > tsh else 1 for s in scores_used['P_worst'][main_key]]
            else:
                raise Exception("Alternative variable does not exist, choose between 'best' or 'worst'")
            
            # mean and std of scores to save
            scores_used_mean['P_correct'] = {k: (np.mean(v), np.std(v)) for k, v in scores_used['P_correct'].items()}
            scores_used_mean['P_best'] = {k: (np.mean(v), np.std(v)) for k, v in scores_used['P_best'].items()}
            scores_used_mean['P_worst'] = {k: (np.mean(v), np.std(v)) for k, v in scores_used['P_worst'].items()}
            scores_used_mean['threshold_best'] = {k: np.mean([scores_used_mean['P_correct'][k][0], scores_used_mean['P_best'][k][0]]) for k in scores_used['P_correct'].keys()}
            scores_used_mean['threshold_worst'] = {k: np.mean([scores_used_mean['P_correct'][k][0], scores_used_mean['P_worst'][k][0]]) for k in scores_used['P_correct'].keys()}

            # compute sign multiplication 
            sign_xor = np.bitwise_xor(signed_main, signed_alt)
            all_signes.append(sign_xor)
            all_values_keys[main_key] = sign_xor


    all_signes = np.concatenate(all_signes)
    return np.mean(all_signes), np.std(all_signes), scores_used_mean, all_values_keys

In [34]:
def score_calculation(kg: str, strat: str, negatives: list[bool], formulas: list[str], relation_pairs: list, max_lmbda: float, save: bool=False):
    for neg in negatives:
        for fr in formulas:
            temp_scores = {}
            for rel_pair in relation_pairs:
                temp_scores[rel_pair] = exe_ps(kg=kg, rels_pair=rel_pair, strategy=strat, formula=fr, negative=neg, max_lmbda=max_lmbda)

            if neg:
                if fr == 'max' or fr == 'comb':
                    score_dir = f"plausibility_scores/{kg}/{strat}/{kg}_{strat}_neg_{fr}_{max_lmbda}_scores.pkl"
                else:
                    score_dir = f"plausibility_scores/{kg}/{strat}/{kg}_{strat}_neg_{fr}_scores.pkl"
            else:
                if fr == 'max' or fr == 'comb':
                    score_dir = f"plausibility_scores/{kg}/{strat}/{kg}_{strat}_blind_{fr}_{max_lmbda}_scores.pkl"
                else:
                    score_dir = f"plausibility_scores/{kg}/{strat}/{kg}_{strat}_blind_{fr}_scores.pkl"

            if save:
                os.makedirs(os.path.dirname(score_dir), exist_ok=True)
                with open(score_dir, "wb") as f:
                    pickle.dump(temp_scores, f)
                print(f'File saved: {score_dir}')


In [35]:
def save_results(scores, kg, name):
    # Save and results of columns
    with open(f'all_score_results/{kg}/{name}.pkl', 'wb') as f:
        pickle.dump(scores, f)


### Execute

In [ ]:
kgs = ['miRNA-KG', 'PKT-KG', 'Hetionet', 'PrimeKG', 'OptimusKG']
kg = 'OptimusKG' 
strategies = ['c-b-n-s', 'r-n-s']
frmls = ['base', 'max', 'comb', 'adj_softmax_2', 'adj_softmax_1', 'adj_softmax_3', 'comb_2', 'comb_3', 'comb_4', 'comb_5', 'comb_6', 'comb_7', 'comb_8']
formulas_dist = ["base_dist", "max_dist", "comb_dist", "adj_softmax_2_dist", "adj_softmax_1_dist", "adj_softmax_3_dist", "comb_2_dist", "comb_3_dist", "comb_4_dist", "comb_5_dist", "comb_6_dist", "comb_7_dist", "comb_8_dist"]
negatives = [True, False]
max_lmbda = 10

if kg == 'PKT-KG':
    relation_pairs = list(PKT_KG_relations.keys())
elif kg == 'miRNA-KG':
    relation_pairs = list(miRNA_KG_relations.keys())
elif kg == 'Hetionet':
    relation_pairs = list(Hetionet_relations.keys())
elif kg == 'PrimeKG':
    relation_pairs = list(PrimeKG_relations.keys())
elif kg == 'OptimusKG':
    relation_pairs = list(OptimusKG_relations.keys())


#### Compute plausibility scores

In [ ]:
for kg in kgs:
    if kg == 'PKT-KG':
        relation_pairs = list(PKT_KG_relations.keys())
    elif kg == 'miRNA-KG':
        relation_pairs = list(miRNA_KG_relations.keys())
    elif kg == 'Hetionet':
        relation_pairs = list(Hetionet_relations.keys())
    elif kg == 'PrimeKG':
        relation_pairs = list(PrimeKG_relations.keys())
    elif kg == 'OptimusKG':
        relation_pairs = list(OptimusKG_relations.keys())

    formula_compute = ['max', 'comb', 'adj_softmax_2', 'adj_softmax_1', 'adj_softmax_3', 'comb_2', 'comb_3', 'comb_4', 'comb_5', 'comb_6', 'comb_7', 'comb_8']

    # Execute to compute scores and save them
    for s in strategies:
        score_calculation(kg, s, negatives, formula_compute, relation_pairs, max_lmbda=max_lmbda, save=True)


    # save blind scores with alternative
    neg = False
    formula_compute_dist = ["base_dist", "max_dist", "comb_dist", "adj_softmax_2_dist", "adj_softmax_1_dist", "adj_softmax_3_dist", "comb_2_dist", "comb_3_dist", "comb_4_dist", "comb_5_dist", "comb_6_dist", "comb_7_dist", "comb_8_dist"]
    for strat in strategies:
        for fr in formula_compute_dist:
            scores = dict()
            score_save_dir = f"p_score_log/{kg}/ps_{fr[:-5]}_{kg}_{strat}.pkl"
            for rp in relation_pairs:
                scores[rp] = exe_ps(kg=kg, rels_pair=rp, strategy=strat, formula=fr, negative=neg, max_lmbda=max_lmbda)
            os.makedirs(os.path.dirname(score_save_dir), exist_ok=True)
            with open(score_save_dir, "wb") as file:
                pickle.dump(scores, file)
            print(f'File saved: {score_save_dir}', color='g')
                
    # save negative scores with alternative
    neg = True
    for strat in strategies:
        for fr in formula_compute_dist:
            scores = dict()
            score_save_dir = f"p_score_log/{kg}/ps_{fr[:-5]}_{kg}_neg_{strat}.pkl"
            # create the scores
            for rp in relation_pairs:
                scores[rp] = exe_ps(kg=kg, rels_pair=rp, strategy=strat, formula=fr, negative=neg, max_lmbda=max_lmbda)
            os.makedirs(os.path.dirname(score_save_dir), exist_ok=True)
            with open(score_save_dir, "wb") as file:
                pickle.dump(scores, file)
            print(f"Saved: {score_save_dir}", color='r')


#### Distance from 0

In [ ]:
# Distance from 0
negative = True
NEG = {}
for strat in strategies:
    NEG[strat] = {}
    for fr in frmls:
        if fr == 'max' or fr == 'comb':
            with open(f"plausibility_scores/{kg}/{strat}/{kg}_{strat}_neg_{fr}_{max_lmbda}_scores.pkl", "rb") as f:
                scores = pickle.load(f)
        else:
            with open(f"plausibility_scores/{kg}/{strat}/{kg}_{strat}_neg_{fr}_scores.pkl", "rb") as f:
                scores = pickle.load(f)

        scores_list = list(scores.values())
        mean, std, NEG[strat][fr] = global_mean_std(scores_list, negative)
        print(f"{kg}, {strat}, {fr}, Negative={negative}:\n{mean:.4f} ± {std:.4f}", color='r')


In [ ]:
save_results(NEG, kg, 'NEG')
print(f'All results saved for {kg}')


#### Distance from 1

In [ ]:
# Distance from 1
negative = False
POS = {}
for strat in strategies:
    POS[strat] = {}
    for fr in frmls:
        if fr == 'max' or fr == 'comb':
            with open(f"plausibility_scores/{kg}/{strat}/{kg}_{strat}_blind_{fr}_{max_lmbda}_scores.pkl", "rb") as f:
                scores = pickle.load(f)
        else:
            with open(f"plausibility_scores/{kg}/{strat}/{kg}_{strat}_blind_{fr}_scores.pkl", "rb") as f:
                scores = pickle.load(f)

        scores_list = list(scores.values())
        mean, std, POS[strat][fr] = global_mean_std(scores_list, negative)
        print(f"{kg}, {strat}, {fr}, Negative={negative}:\n{mean:.4f} ± {std:.4f}", color='g')


In [ ]:
save_results(POS, kg, 'POS')
print(f'All results saved for {kg}')


#### AVG Distance from 1 and 0

In [ ]:
# Average distance from 0 and 1
#* 3
AVG = {}
for strat in strategies:
    AVG[strat] = {}
    for fr in frmls:
        if fr == 'max' or fr == 'comb':
            with open(f"plausibility_scores/{kg}/{strat}/{kg}_{strat}_blind_{fr}_{max_lmbda}_scores.pkl", "rb") as f:
                scores_blind = pickle.load(f)
            with open(f"plausibility_scores/{kg}/{strat}/{kg}_{strat}_neg_{fr}_{max_lmbda}_scores.pkl", "rb") as f:
                scores_neg = pickle.load(f)
        else:
            with open(f"plausibility_scores/{kg}/{strat}/{kg}_{strat}_blind_{fr}_scores.pkl", "rb") as f:
                scores_blind = pickle.load(f)
            with open(f"plausibility_scores/{kg}/{strat}/{kg}_{strat}_neg_{fr}_scores.pkl", "rb") as f:
                scores_neg = pickle.load(f)
        
        scores_blind_list = list(scores_blind.values())
        scores_neg_list = list(scores_neg.values())
        mean, std, AVG[strat][fr] = global_mean_std_combined(scores_blind_list, scores_neg_list)
        print(f"{kg}, {strat}, {fr}:\n{mean:.4f} ± {std:.4f}", color='c')


In [ ]:
save_results(AVG, kg, 'AVG')
print(f'All results saved for {kg}')


#### Main vs Alternative

In [ ]:
# Calculate distance main plausibility from alternative ****
neg = False
COM = {}
for strat in strategies:
    COM[strat] = {}
    for fr in formulas_dist:
        scores = dict()
        score_save_dir = f"p_score_log/{kg}/ps_{fr[:-5]}_{kg}_{strat}.pkl"
        if not os.path.exists(score_save_dir):
            for rp in relation_pairs:
                scores[rp] = exe_ps(kg=kg, rels_pair=rp, strategy=strat, formula=fr, negative=neg, max_lmbda=max_lmbda)
            os.makedirs(os.path.dirname(score_save_dir), exist_ok=True)
            with open(score_save_dir, "wb") as file:
                pickle.dump(scores, file)
        else:
            with open(score_save_dir, "rb") as file:
                scores = pickle.load(file)

        scores_list = [scores[k] for k in scores.keys()]
        mean, std, COM[strat][fr] = global_mean_std_main_vs_alts(scores_list)
        print(f"{kg}, {strat}, {fr}:\n{mean:.4f} ± {std:.4f}", color="y")


In [ ]:
save_results(COM, kg, 'COM')
print(f'All results saved for {kg}')


#### Correct / Best (0.5 Threshold)

In [ ]:
#  correct vs best *****
neg = False
su = dict()
BEST = {}
for strat in strategies:
    su[strat] = {}
    BEST[strat] = {}
    for f in formulas_dist:
        su[strat][f[:-5]] = dict()
        scores = dict()
        score_save_dir = f"p_score_log/{kg}/ps_{f[:-5]}_{kg}_{strat}.pkl"
        if not os.path.exists(score_save_dir):
            # create the scores
            for rp in relation_pairs:
                scores[rp] = exe_ps(kg=kg, rels_pair=rp, strategy=strat, formula=f, negative=neg, max_lmbda=max_lmbda)
            # save the scores
            os.makedirs(os.path.dirname(score_save_dir), exist_ok=True)
            with open(score_save_dir, "wb") as file:
                pickle.dump(scores, file)
        else:
            # load the scores
            with open(score_save_dir, "rb") as file:
                scores = pickle.load(file)

        scores_list = [scores[k] for k in scores.keys()]
        mean, std, su[strat][f[:-5]], BEST[strat][f] = avg_sign(scores_list, 'best')
        print(f"{kg}, {strat}, {f}, correct vs best (0.5):\n {mean:.4f} ± {std:.4f}", background='m', color='k')


In [ ]:
save_results(BEST, kg, 'BEST')
print(f'All results saved for {kg}')


#### Correct / Worst (0.5 Threshold)

In [ ]:
# correct vs worst ******
neg = False
su = dict()
WORST = {}
for strat in strategies:
    su[strat] = {}
    WORST[strat] = {}
    for f in formulas_dist:
        su[strat][f[:-5]] = dict()
        scores = dict()
        score_save_dir = f"p_score_log/{kg}/ps_{f[:-5]}_{kg}_{strat}.pkl"
        if not os.path.exists(score_save_dir):
            # create the scores
            for rp in relation_pairs:
                scores[rp] = exe_ps(kg=kg, rels_pair=rp, strategy=strat, formula=f, negative=neg, max_lmbda=max_lmbda)
            # save the scores
            os.makedirs(os.path.dirname(score_save_dir), exist_ok=True)
            with open(score_save_dir, "wb") as file:
                pickle.dump(scores, file)
        else:
            # load the scores
            with open(score_save_dir, "rb") as file:
                scores = pickle.load(file)

        scores_list = [scores[k] for k in scores.keys()]
        mean, std, su[strat][f[:-5]], WORST[strat][f] = avg_sign(scores_list, 'worst')
        print(f"{kg}, {strat}, {f}, correct vs worst (0.5):\n {mean:.4f} ± {std:.4f}", background='y', color='k')


In [ ]:
save_results(WORST, kg, 'WORST')
print(f'All results saved for {kg}')


In [ ]:
# Save correct_best_worst with fixed thresholds 
cbw_save_dir = f"correct_best_worst/{v}_correct_best_worst.pkl"
os.makedirs(os.path.dirname(cbw_save_dir), exist_ok=True)
with open(cbw_save_dir, "wb") as file:
    pickle.dump(su, file)
print(f"Saved: {cbw_save_dir}", color='g')

#### Correct / Best (mean threshold)

In [ ]:
# "Correct" vs Best Calibrated Thr. 7*
neg = False
BEST_balanced = {}
for strat in ['c-b-n-s', 'r-n-s']:
    BEST_balanced[strat] = {}
    for f in formulas_dist:
        scores = dict()
        score_save_dir = f"p_score_log/{kg}/ps_{f[:-5]}_{kg}_{strat}.pkl"
        if not os.path.exists(score_save_dir):
            # create the scores
            for rp in relation_pairs:
                scores[rp] = exe_ps(kg=kg, rels_pair=rp, strategy=strat, formula=f, negative=neg, max_lmbda=max_lmbda)
            # save the scores
            os.makedirs(os.path.dirname(score_save_dir), exist_ok=True)
            with open(score_save_dir, "wb") as file:
                pickle.dump(scores, file)
        else:
            # load the scores
            with open(score_save_dir, "rb") as file:
                scores = pickle.load(file)

        # read correct_best_worst
        with open(f"correct_best_worst/{kg}_correct_best_worst.pkl", "rb") as file:
            cbw = pickle.load(file)
         
        scores_list = [scores[k] for k in scores.keys()]
        mean, std, su, BEST_balanced[strat][f] = avg_sign(scores_list, 'best', cbw[strat][f[:-5]]['threshold_best'])
        print(f"{kg}, {strat}, {f},  correct vs best (mean):\n {mean:.4f} ± {std:.4f}", background='c', color='k')


In [ ]:
save_results(BEST_balanced, kg, 'BEST_balanced')
print(f'All results saved for {kg}')


#### Correct / Worst (mean threshold)

In [ ]:
# "Correct" vs Best Calibrated Thr. 8*
neg = False
WORST_balanced = {}
for strat in strategies:
    WORST_balanced[strat] = {}
    for f in formulas_dist:
        scores = dict()
        score_save_dir = f"p_score_log/{kg}/ps_{f[:-5]}_{kg}_{strat}.pkl"
        if not os.path.exists(score_save_dir):
            # create the scores
            for rp in relation_pairs:
                scores[rp] = exe_ps(kg=kg, rels_pair=rp, strategy=strat, formula=f, negative=neg, max_lmbda=max_lmbda)
            # save the scores
            os.makedirs(os.path.dirname(score_save_dir), exist_ok=True)
            with open(score_save_dir, "wb") as file:
                pickle.dump(scores, file)
        else:
            # load the scores
            with open(score_save_dir, "rb") as file:
                scores = pickle.load(file)

        # read correct_best_worst
        with open(f"correct_best_worst/{kg}_correct_best_worst.pkl", "rb") as file:
            cbw = pickle.load(file)
         
        scores_list = [scores[k] for k in scores.keys()]
        mean, std, su, WORST_balanced[strat][f] = avg_sign(scores_list, 'worst', cbw[strat][f[:-5]]['threshold_worst'])
        print(f"{kg}, {strat}, {f}, correct vs worst (mean):\n {mean:.4f} ± {std:.4f}", background='r', color='w')


In [ ]:
save_results(WORST_balanced, kg, 'WORST_balanced')
print(f'All results saved for {kg}')


### Export Results to CSV

In [ ]:
kgs = ['miRNA-KG', 'PKT-KG', 'Hetionet', 'PrimeKG', 'OptimusKG']
strategies = ['c-b-n-s', 'r-n-s']
frmls = ['base', 'max', 'comb', 'adj_softmax_2', 'adj_softmax_1', 'adj_softmax_3', 'comb_2', 'comb_3', 'comb_4', 'comb_5', 'comb_6', 'comb_7', 'comb_8']

def flatten_simple(vals):
    # scalar = precomputed mean for a mismatched relation; include it directly
    return np.concatenate([np.array([float(v)]) if np.ndim(v) == 0 else v for v in vals.values()])

def flatten_avg(pos_vals, neg_vals):
    # replicates global_mean_std_combined: concat ALL raw pos and neg error arrays
    arrays = [np.atleast_1d(v) for v in pos_vals.values()] + [np.atleast_1d(v) for v in neg_vals.values()]
    return np.concatenate(arrays)

def flatten_com(vals):
    arrays = [arr for inner in vals.values() for arr in inner.values()]
    return np.concatenate(arrays) if arrays else np.array([])

def flatten_sign(vals):
    return np.concatenate([np.array([float(v)]) if np.ndim(v) == 0 else v for v in vals.values()])

os.makedirs('csv_results', exist_ok=True)

for kg in kgs:
    base_dir = f'all_score_results/{kg}'
    if not os.path.exists(base_dir):
        print(f'Skipping {kg}: no results found')
        continue

    metrics = {}
    for name in ['NEG', 'POS', 'AVG', 'COM', 'BEST', 'WORST', 'BEST_balanced', 'WORST_balanced']:
        path = f'{base_dir}/{name}.pkl'
        if os.path.exists(path):
            with open(path, 'rb') as f:
                metrics[name] = pickle.load(f)
        else:
            metrics[name] = None

    rows = []
    for strat in strategies:
        for fr in frmls:
            fr_dist = fr + '_dist'
            row = {'view': kg, 'strategy': strat, 'formula': fr}

            for metric_name, flatten_fn, key in [
                ('NEG',            flatten_simple, fr),
                ('POS',            flatten_simple, fr),
                ('COM',            flatten_com,    fr_dist),
                ('BEST',           flatten_sign,   fr_dist),
                ('WORST',          flatten_sign,   fr_dist),
                ('BEST_balanced',  flatten_sign,   fr_dist),
                ('WORST_balanced', flatten_sign,   fr_dist),
            ]:
                data = metrics.get(metric_name)
                try:
                    vals = data[strat][key]
                    arr = flatten_fn(vals)
                    mean = round(float(np.mean(arr)), 4)
                    std  = round(float(np.std(arr)),  4)
                    row[metric_name] = f"{mean} ± {std}"
                except (KeyError, TypeError, ValueError):
                    row[metric_name] = None

            # AVG: recompute from raw POS and NEG to match notebook output
            try:
                arr = flatten_avg(metrics["POS"][strat][fr], metrics["NEG"][strat][fr])
                mean = round(float(np.mean(arr)), 4)
                std  = round(float(np.std(arr)),  4)
                row["AVG"] = f"{mean} ± {std}"
            except (KeyError, TypeError, ValueError):
                row["AVG"] = None

            rows.append(row)

    csv_path = f'csv_results/{kg}.csv'
    if rows:
        fieldnames = ['view', 'strategy', 'formula', 'NEG', 'POS', 'AVG', 'COM', 'BEST', 'WORST', 'BEST_balanced', 'WORST_balanced']
        with open(csv_path, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        print(f'Saved {csv_path} ({len(rows)} rows)')
